In [1]:
### 03_전처리하기

In [2]:
### 1. 타겟 변수 전처리

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('./input/adult.csv')

In [5]:
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [6]:
# 타겟 변수 income을 검토하고 전처리
# income은 조사 응답자의 연 소득이 5만 달러를 초과하는지 여부를 나타냄

In [8]:
df['income'].value_counts(normalize=True) # 수가 아닌 비율로 나타내줌

income
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64

In [9]:
# 연 소득이 5만 달러를 초과하는 사람이 23.9%, 5만 달러 이하인 사람이 76.1%

# 변수의 값에 특수 문자나 대소문자가 섞여 있으면 다루기 불편하므로 5만 달러 초과하면 'high',
# 그렇지 않으면 'low'로 값을 수정

In [10]:
import numpy as np
df['income'] = np.where(df['income'] == '>50K', 'high', 'low')
df['income'].value_counts(normalize=True)

income
low     0.760718
high    0.239282
Name: proportion, dtype: float64

In [11]:
### 2. 불필요한 변수 제거하기

In [12]:
# 이름, 아이디, 주소 같은 변수는 대부분 값이 고유값이어서 반복되는 패턴이 없고 타켓 변수와도 관련성이 없음
# 이런 변수는 타겟 변수를 예측하는데 도움이 되지 않고 모델링 시간만 늘이는 역할을 하므로 제거
#
# fnlwgt는 adult 데이터를 이용해 미국의 실제 인구를 추정할 때 사용하는 가중치
# 타겟 변수를 예측하는데 도움이 되지 않기 때문에 제거

In [14]:
df = df.drop(columns='fnlwgt')

In [15]:
### 3. 문자 타입 변수를 숫자 타입으로 바꾸기

In [16]:
# 모델을 만드는데 사용되는 모든 변수는 숫자 타입이어야 함
# df.info() 실행 결과 Dtype이 object인 문자 타입 변수들이 있는데,
# 이 변수들을 모델에 활용하려면 숫자 타입으로 바꿔야 함

In [17]:
# 1) 원 핫 인코딩하기

In [18]:
# 원핫 인코딩 one-hot-encoding : 값을 1과 0으로 바꾸는 방법
# 변수의 범주가 특정 값이면 1, 그렇지 않으면 0으로 바꾸어 문자 타입을 숫자 타입으로 만들 수 있음

In [19]:
# 성별을 추출해 원핫 인코딩
df_tmp = df[['sex']]
df_tmp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   sex     48842 non-null  object
dtypes: object(1)
memory usage: 381.7+ KB


In [20]:
df_tmp.value_counts(normalize=True)

sex   
Male      0.668482
Female    0.331518
Name: proportion, dtype: float64

In [21]:
# 출력 결과를 보면 sex는 Male, Female로 되어있는 문자열 타입 변수

In [22]:
df_tmp.head()

,sex
0,Male
1,Male
2,Male
3,Male
4,Female


In [23]:
# pd.get_dummies()에 데이터 프레임을 입력하면 문자 타입 변수를 원핫 인코딩을 적용해 변환
# df_tmp의 문자 타입 변수에 원핫 인코딩 적용
df_tmp = pd.get_dummies(df_tmp, dtype=int)
df_tmp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   sex_Female  48842 non-null  int64
 1   sex_Male    48842 non-null  int64
dtypes: int64(2)
memory usage: 763.3 KB


In [24]:
df_tmp.head()

,sex_Female,sex_Male
0,0,1
1,0,1
2,0,1
3,0,1
4,1,0


In [25]:
# 출력 결과를 보면 sex가 사라지고 그 대신 sex_Female, sex_Male이 만들어 짐
# 원핫 인코딩으로 만들어진 변수의 타입은 int32

In [26]:
# sex_Female은 sex가 Female이면 1, 그렇지 않으면 0으로 된 변수
# sex_Male은 반대로 sex가 Male이면 1, 그렇지 않으면 0으로 된 변수
# 두 변수 중 한쪽이 1이면 다른 한쪽은 반드시 0

In [27]:
# df에 원핫 인코딩 적용
# 타겟 변수인 income만 원래대로 유지하고, 모든 문자 타입 변수를 원핫 인코딩 적용
target = df['income'] # income 추출

In [28]:
df = df.drop(columns='income') # income 제거

In [29]:
df = pd.get_dummies(df, dtype=int) # 문자 타입 변수 원핫 인코딩

In [30]:
df['income'] = target # df에 target 삽입

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Columns: 108 entries, age to income
dtypes: int64(107), object(1)
memory usage: 40.2+ MB


In [32]:
# df.info() 출력 결과에 개별 변수의 정보가 출력되지 않은 이유는
# 변수가 100개 이하일 때만 뱐수 정보를 출력하도록 설정되어 있기 때문
# df.info()에 max_cols=np.inf를 입력하면 변수의 수와 관계없이 모든 변수의 정보를 출력

In [33]:
import numpy as np
df.info(max_cols=np.inf) # infinite (무한대로 열 출력)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 108 columns):
 #    Column                                     Non-Null Count  Dtype 
---   ------                                     --------------  ----- 
 0    age                                        48842 non-null  int64 
 1    education_num                              48842 non-null  int64 
 2    capital_gain                               48842 non-null  int64 
 3    capital_loss                               48842 non-null  int64 
 4    hours_per_week                             48842 non-null  int64 
 5    workclass_?                                48842 non-null  int64 
 6    workclass_Federal-gov                      48842 non-null  int64 
 7    workclass_Local-gov                        48842 non-null  int64 
 8    workclass_Never-worked                     48842 non-null  int64 
 9    workclass_Private                          48842 non-null  int64 
 10   workclass_Self-emp-i

In [34]:
# 출력 결과를 보면 변수가 108개로 늘어나고, 문자 타입 변수가 전부 숫자 타입으로 바ㅏ뀐 것을 확인

In [35]:
df.head()

,age,education_num,capital_gain,capital_loss,hours_per_week,workclass_?,workclass_Federal-gov,workclass_Local-gov,workclass_Never-worked,workclass_Private,...,native_country_Puerto-Rico,native_country_Scotland,native_country_South,native_country_Taiwan,native_country_Thailand,native_country_Trinadad&Tobago,native_country_United-States,native_country_Vietnam,native_country_Yugoslavia,income
0,25,7,0,0,40,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,low
1,38,9,0,0,50,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,low
2,28,12,0,0,40,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,high
3,44,10,7688,0,40,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,high
4,18,10,0,0,30,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,low


In [36]:
df.to_pickle('./output/df_step_01.pickle') # pickle: 2진(binary) 형식